In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import gc

import itertools
from scipy.stats import ttest_ind

import sklearn.model_selection
import sklearn.linear_model
import scipy.stats

#from act_max_util import *

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:
def load_data_set(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index =  subject_index
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    print(all_epochs.shape)
    
    return all_epochs, labels_raw, ch_names

In [ ]:
dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/random_test"
freq_bands = ["delta", "theta", "alpha", "beta", "gamma"]
layers=['trunk_net.spatial_filter.layers.conv_temporal', 'trunk_net.spatial_filter.layers.conv_spatial', 'trunk_net.spatial_filter.layers.conv_separable_point',
         'trunk_net.spatial_filter.layers.conv_separable_depth','trunk_net.spatial_filter' , 
        'trunk_net.s4_blocks.0', 'cross_trial_s4', 'head_net.fc_mean1']
        #'trunk_net.s4_blocks.0']
ignore_channels = ['Cz', 'Iz', 'Fz', 'Oz', 'Pz']

In [ ]:
def load_RCAV_results(rep=0):
    load_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/results_new_generator_norm_non_abs3"
    load_file = f"RCAV_band_power_results_subject_2_{rep}.npy"
    data = np.load(f"{load_dir}/{load_file}", allow_pickle=True).item()

    return data

In [ ]:
def load_result(kind="scores"):
    all_data = {mi:{freq_band:{ch_name : {layer: {kind:[], "n":0,} for layer in layers} for ch_name in ch_names} for freq_band in freq_bands} for mi in range(100,501,100)}
    for rep in range(9):
        data = load_RCAV_results(rep)
        model_idx = np.arange(100,501,100)
        for mi in model_idx:
            for freq_band in freq_bands:
                for ch_name in ch_names:
                    for layer in layers:
                        if data[mi][freq_band][ch_name][layer] is not None:
                            all_data[mi][freq_band][ch_name][layer][kind].append(data[mi][freq_band][ch_name][layer][kind])
                            all_data[mi][freq_band][ch_name][layer]["n"]+=1
    for mi in model_idx:
        for freq_band in freq_bands:
            for ch_name in ch_names:
                for layer in layers:
                    if all_data[mi][freq_band][ch_name][layer]["n"] >=9:
                        all_data[mi][freq_band][ch_name][layer][kind] = np.nanmean(np.array(all_data[mi][freq_band][ch_name][layer][kind]))
    return all_data
    

In [ ]:
def load_result2(kind="scores"):
    all_data = {mi:{freq_band:{ch_name : {layer: {"all":[], "n":0, kind:0} for layer in layers} for ch_name in ch_names} for freq_band in freq_bands} for mi in range(100,501,100)}
    for rep in range(9):
        data = load_RCAV_results(rep)
        model_idx = np.arange(100,501,100)
        for mi in model_idx:
            for freq_band in freq_bands:
                for ch_name in ch_names:
                    for layer in layers:
                        if data[mi][freq_band][ch_name][layer] is not None:
                            all_data[mi][freq_band][ch_name][layer]["all"].append(data[mi][freq_band][ch_name][layer][kind])
                            all_data[mi][freq_band][ch_name][layer]["n"]+=1
    for mi in model_idx:
        for freq_band in freq_bands:
            for ch_name in ch_names:
                for layer in layers:
                    if all_data[mi][freq_band][ch_name][layer]["n"] >=9:
                        all_data[mi][freq_band][ch_name][layer][kind] = np.nanmean(np.array(all_data[mi][freq_band][ch_name][layer]["all"]))
    return all_data

In [ ]:
all_epochs, labels_raw, ch_names = load_data_set(2)

In [ ]:
Br_scores = load_result("Br")

In [ ]:
R2 = load_result("mean_score")

In [ ]:
R2_2 = load_result2("mean_score")

In [ ]:
layer_labels = ["TemporalConv", "SpatialConv", "SeparablePointConv", "SeparableDepthConv", "EEGNet", "S4Block", "S4Block2", "HeadNet FC1"]

In [ ]:
def plot_results(result, layer, kind="best_score"):
    model_indices = list(range(100, 501, 100))
    
    # Create subplot grid: rows for model indices, columns for frequency bands
    fig, axes = plt.subplots(len(model_indices), len(freq_bands), 
                            figsize=(20, 4*len(model_indices)), 
                            sharex=True, sharey=True)
    
    fig.suptitle(f'Distribution of {kind} scores for layer: {layer}', fontsize=16)
    fig.subplots_adjust(hspace=0.3, wspace=0.1)

    # Plot histograms for each combination of model index and frequency band
    for i, mi in enumerate(model_indices):
        for j, freq_band in enumerate(freq_bands):
            ax = axes[i, j]
            
            # Collect values across all channels for this model index and frequency band
            values = []
            for ch_name in ch_names:
                if ch_name not in ignore_channels:
                    val = result[mi][freq_band][ch_name][layer][kind]
                    if isinstance(val, (int, float)):
                        values.append(val)
            
            # Create histogram
            if values:
                ax.hist(values, bins=20, edgecolor='black')
                ax.axvline(np.mean(values), color='r', linestyle='dashed', 
                          label=f'Mean: {np.mean(values):.3f}')
                ax.legend()
            
            # Add labels
            if i == len(model_indices)-1:  # Bottom row
                ax.set_xlabel(freq_band)
            if j == 0:  # Leftmost column
                ax.set_ylabel(f'MI={mi}')

    plt.show()


# separate plot for each frequenyc band and model index. In each subplot show distribution of Br scores, with layer color coded?

In [ ]:
def plot_layer_scores(result, model_index=100, kind="Br"):
    # Create a figure with subplots for each frequency band
    fig, axes = plt.subplots(1, len(freq_bands), figsize=(20, 4), sharey=True)
    fig.suptitle(f'Average {kind} scores across channels for model index {model_index}', fontsize=16)
    
    # Process each frequency band
    for freq_idx, freq_band in enumerate(freq_bands):
        ax = axes[freq_idx]
        
        # Calculate average scores across channels for each layer
        layer_scores = []
        for layer in layers:
            scores = []
            for ch_name in ch_names:
                if ch_name not in ignore_channels:
                    val = result[model_index][freq_band][ch_name][layer][kind]
                    if isinstance(val, (int, float)):
                        scores.append(val)
            if scores:
                layer_scores.append(np.mean(scores))
            else:
                layer_scores.append(0)
        
        # Create bar plot
        x = range(len(layers))
        bars = ax.bar(x, layer_scores)
        
        # Customize plot
        ax.set_title(f'{freq_band}')
        ax.set_xticks(x)
        # Use shortened layer names for better readability
        layer_labels = [layer.split('.')[-1] if '.' in layer else layer.split('_')[-1] for layer in layers]
        ax.set_xticklabels(layer_labels, rotation=45, ha='right')
        
        # Add value labels on top of bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}',
                   ha='center', va='bottom', rotation=90)
    
    # Add common ylabel
    fig.text(0.04, 0.5, f'Average {kind} Score', va='center', rotation='vertical')
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_layer_scores_all(result, kind="Br"):
    # Create a figure with subplots for each frequency band
    fig, axes = plt.subplots(len(freq_bands), 1, figsize=(16, 2.5*len(freq_bands)), sharex=True)
    fig.suptitle(f'Average R2 score for different models and layers', fontsize=20, y=1.005)
    
    # Get all model indices
    model_indices = list(range(100, 401, 100))
    #model_indices =[200]
    # Create color map for different model indices
    colors = plt.cm.viridis(np.linspace(0, 1, len(model_indices)))
    
    # Process each frequency band
    for freq_idx, freq_band in enumerate(freq_bands):
        ax = axes[freq_idx]
        
        # Plot lines for each model index
        for idx, model_index in enumerate(model_indices):
            # Calculate average scores across channels for each layer
            layer_scores = []
            for layer in layers:
                scores = []
                for ch_name in ch_names:
                    if ch_name not in ignore_channels:
                        val = result[model_index][freq_band][ch_name][layer][kind]
                        if isinstance(val, (int, float)):
                            scores.append(val)
                if scores:
                    layer_scores.append(np.mean(scores))
                else:
                    layer_scores.append(0)
            
            # Create line plot
            x = range(len(layers))
            line = ax.plot(x, layer_scores, marker='o', linewidth=2, markersize=8, 
                         label=f'Time index={model_index-100}', color=colors[idx], alpha=0.8)
            
        # Customize plot
        ax.set_title(f'{freq_band}', fontsize=18)
        ax.set_xticks(x)
        # Use shortened layer names for better readability
        #layer_labels = [layer.split('.')[-1] if '.' in layer else layer.split('_')[-1] for layer in layers]
        ax.set_xticklabels(layer_labels, rotation=30, ha='right', fontsize=15)
        ax.tick_params(axis='y', labelsize=15)
        #ax.set_ylabel(f'Average {kind} Score', fontsize=16)
        if freq_idx == len(freq_bands) - 1:
            ax.set_xlabel('Layers', fontsize=16)
        ax.set_ylim(0, 1.2)
        
        # Add grid for better readability
        #ax.grid(True, linestyle='--', alpha=0.7)
        
        # Add legend only in the first subplot
        if freq_idx == 0:
            ax.legend(bbox_to_anchor=(0.01, 0.95), loc='upper left', fontsize=15)
    
    # Add common ylabel
    fig.text(0.001, 0.5, f'Average R2 Score', va='center', rotation='vertical', fontsize=18)
    
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
plot_layer_scores_all(R2, kind="mean_score")

In [ ]:
fig = plot_layer_scores_all(R2_2, kind="mean_score")

In [ ]:
fig.savefig(f"R2.png", dpi=300, bbox_inches='tight')

In [ ]:
def plot_channel_layer_scores(channel_name, freq_band, kind="Br", use_abs=False):
    """
    Plot scores across different layers for a specific channel and frequency band,
    comparing different model indices.
    
    Args:
        channel_name (str): Name of the channel to analyze
        freq_band (str): Frequency band to analyze ('delta', 'theta', 'alpha', 'beta', 'gamma')
        kind (str): Type of score to plot ('Br' or 'mean_score')
        use_abs (bool): Whether to use absolute values of scores
    """
    # Get model indices
    model_indices = list(range(100, 401, 100))
    
    # Create figure
    #plt.figure(figsize=(12, 6))
    fig,ax = plt.subplots(figsize=(16, 4.5))
    # Create color map for different model indices
    colors = plt.cm.viridis(np.linspace(0, 1, len(model_indices)))
    
    # Collect scores for each model index
    bar_width = 0.15  # Width of each bar
    for idx, mi in enumerate(model_indices):
        scores = []
        for layer in layers:
            val = Br_scores[mi][freq_band][channel_name][layer][kind]
            if isinstance(val, (int, float)):
                scores.append(abs(val) if use_abs else val)
            else:
                scores.append(0)
        
        # Plot bars with offset
        x_positions = np.arange(len(layers)) + idx * bar_width
        ax.bar(x_positions, scores, width=bar_width, label=f'Time index={mi-100}', 
                alpha=0.7, color=colors[idx])
    
    # Adjust x-axis position to center the groups of bars
    ax.set_xticks(np.arange(len(layers)) + bar_width * (len(model_indices)-1)/2, 
               [layer.split('.')[-1] if '.' in layer else layer.split('_')[-1] for layer in layers], 
               rotation=30, ha='right', fontsize=14)
    ax.set_xticklabels(layer_labels, rotation=30, ha='right', fontsize=15)
    
    # Customize plot
    score_type = f"Absolute {kind} Scores" if use_abs else f"{kind} Scores"
    ax.set_title(f'{score_type} Across Layers for {channel_name} ({freq_band} band)', fontsize=18)
    ax.set_xlabel('Layers', fontsize=16)
    ax.set_ylabel(score_type, fontsize=16)
    ax.tick_params(axis='both', labelsize=15)
    
    # Add grid for better readability
    plt.grid(True, alpha=0.3)
    
    # Add legend
    plt.legend(bbox_to_anchor=(0.05, 1), loc='upper left', fontsize=14)
    
    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    
    plt.show()

    return fig

In [ ]:
fig = plot_channel_layer_scores("O2", "gamma")

In [ ]:
fig.savefig(f"O2_gamma.png", dpi=300, bbox_inches='tight')

In [ ]:
#plot_channel_layer_scores("C4", "gamma")

In [ ]:
#plot_layer_scores_all(Br_scores, kind="Br")

In [ ]:
#for band in freq_bands:
#    for ch_name in ch_names:
#        if ch_name not in ignore_channels:
#            plot_channel_layer_scores(ch_name, band)

In [ ]:
def plot_score_hist(result, layer, kind="mean_score", use_abs=False):
    fig, axes = plt.subplots(1, len(freq_bands), figsize=(20, 4))
    fig.suptitle(f'Distribution of {kind} scores for layer: {layer}', fontsize=16)
    
    model_indices = list(range(100, 501, 100))
    
    for freq_idx, freq_band in enumerate(freq_bands):
        scores = []
        # Collect scores for all channels and model indices
        for mi in model_indices:
            for ch_name in ch_names:
                if ch_name not in ignore_channels:
                    if use_abs:
                        val = np.abs(result[mi][freq_band][ch_name][layer][kind])
                    else:
                        val = result[mi][freq_band][ch_name][layer][kind]

                    
                    if isinstance(val, (int, float)):
                        scores.append(val)
        
        if scores:
            # Create histogram
            ax = axes[freq_idx]
            ax.hist(scores, bins=20, density=True)
            ax.set_title(f'{freq_band}')
            ax.set_xlabel(f'{kind} score')
            
            # Add mean line
            mean_val = np.mean(scores)
            ax.axvline(mean_val, color='r', linestyle='dashed', 
                      label=f'Mean: {mean_val:.3f}')
            ax.legend()
            
            # Add grid for better readability
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_score_heatmap(result, layer, kind="mean_score"):
    # Get model indices
    model_indices = list(range(100, 501, 100))
    
    # Calculate number of rows needed for 2 plots per row
    num_rows = int(np.ceil(len(model_indices) / 2))
    
    # Create a figure with subplots - 2 plots per row
    fig, axes = plt.subplots(num_rows, 2, figsize=(15, 5*num_rows))
    fig.suptitle(f'{kind} scores for layer: {layer}', fontsize=16)
    fig.subplots_adjust(hspace=0.3, wspace=0.2)
    
    # Make axes 2D if it's not already
    if num_rows == 1:
        axes = axes.reshape(1, -1)
    
    # Remove ignored channels
    channels = [ch for ch in ch_names if ch not in ignore_channels]
    
    # Plot heatmaps
    for idx, mi in enumerate(model_indices):
        row = idx // 2
        col = idx % 2
        
        # Create data matrix for heatmap
        data_matrix = np.zeros((len(freq_bands), len(channels)))
        
        # Fill the matrix with scores
        for i, freq_band in enumerate(freq_bands):
            for j, ch_name in enumerate(channels):
                val = result[mi][freq_band][ch_name][layer][kind]
                if isinstance(val, (int, float)):
                    data_matrix[i, j] = val
                else:
                    data_matrix[i, j] = np.nan
        
        # Create heatmap
        im = axes[row, col].imshow(data_matrix, aspect='auto', cmap='RdYlBu_r')
        
        # Customize plot
        axes[row, col].set_title(f'Model Index = {mi}')
        axes[row, col].set_xticks(range(len(channels)))
        axes[row, col].set_yticks(range(len(freq_bands)))
        axes[row, col].set_xticklabels(channels, rotation=90)
        axes[row, col].set_yticklabels(freq_bands)
        
        # Add colorbar
        plt.colorbar(im, ax=axes[row, col])
    
    # Remove empty subplots if any
    if len(model_indices) % 2 != 0:
        axes[num_rows-1, -1].remove()
    
    plt.tight_layout()
    plt.show()

In [ ]:
layers

In [ ]:
plot_score_hist(R2, 'trunk_net.spatial_filter.layers.conv_separable_depth', kind="mean_score")

In [ ]:
for layer in layers:

    plot_score_hist(Br_scores, layer, kind="Br", use_abs=True)


In [ ]:
layers

In [ ]:
plot_score_heatmap(Br_scores, "trunk_net.s4_blocks.0", kind="Br")

In [ ]:
plot_score_heatmap(Br_scores,  'cross_trial_s4', kind="Br")

In [ ]:
plot_score_heatmap(R2, "trunk_net.s4_blocks.0", kind="mean_score")

In [ ]:
plot_score_heatmap(Br_scores, "trunk_net.s4_blocks.0", kind="Br")

In [ ]:
layers

In [ ]:
plot_score_heatmap(Br_scores, "cross_trial_s4", kind="Br")

In [ ]:
plot_score_heatmap(Br_scores, 'head_net.fc_mean1', kind="Br")

In [ ]:
# add absolute Br scores over all channels for  frequency band for a given layer

In [ ]:
def plot_layer_heatmap(result, layer, kind="Br"):
    # Get model indices and frequency bands
    model_indices = list(range(100, 501, 100))
    
    # Create data matrix for heatmap
    data_matrix = np.zeros((len(freq_bands), len(model_indices)))
    
    # Calculate average absolute scores for each frequency band and model index
    for i, freq_band in enumerate(freq_bands):
        for j, mi in enumerate(model_indices):
            scores = []
            for ch_name in ch_names:
                if ch_name not in ignore_channels:
                    val = result[mi][freq_band][ch_name][layer][kind]
                    if isinstance(val, (int, float)):
                        scores.append(abs(val))
            if scores:
                data_matrix[i,j] = np.mean(scores)
                
    # Create heatmap
    plt.figure(figsize=(10,6))
    plt.imshow(data_matrix, aspect='auto', cmap='RdYlBu_r')
    
    # Add colorbar
    plt.colorbar(label=f'Average absolute {kind} score')
    
    # Customize axes
    plt.yticks(range(len(freq_bands)), freq_bands)
    plt.xticks(range(len(model_indices)), model_indices)
    plt.xlabel('Model Index')
    plt.ylabel('Frequency Band')
    
    plt.title(f'Average absolute {kind} scores for layer: {layer}')
    
    plt.tight_layout()
    plt.show()


In [ ]:
plot_layer_heatmap(Br_scores, "trunk_net.s4_blocks.0", kind="Br")

In [ ]:
#plot_layer_heatmap(Br_scores, "trunk_net.s4_blocks.0", kind="Br")
plot_layer_heatmap(Br_scores, "cross_trial_s4", kind="Br")


In [ ]:
for layer in layers:
    plot_layer_heatmap(Br_scores, layer, kind="Br")


In [ ]:
for layer in layers:
    plot_layer_heatmap(R2, layer, kind="mean_score")


In [ ]:
# nest is topomap plot that shows Br scores. function takes layer as input, makes separate topomap for each frequency band and model index
# also possible to make 
# next show channels per frequency ordered by their absolute Br score.#
#generate channel importances/influences

In [ ]:
def plot_channel_importance(scores, layer, n_top=10):
    # Create subplots for each frequency band
    fig, axes = plt.subplots(1, len(freq_bands), figsize=(20, 5))
    fig.suptitle(f'Channel importance based on absolute Br scores for layer: {layer}', fontsize=16)
    
    # Process each frequency band
    for freq_idx, freq_band in enumerate(freq_bands):
        # Aggregate absolute Br scores for each channel
        channel_scores = {}
        for ch_name in ch_names:
            if ch_name not in ignore_channels:
                abs_scores = []
                for mi in range(100, 501, 100):
                    val = scores[mi][freq_band][ch_name][layer]['Br']
                    if isinstance(val, (int, float)):
                        abs_scores.append(abs(val))
                if abs_scores:
                    channel_scores[ch_name] = np.mean(abs_scores)
        
        # Sort channels by their scores
        sorted_channels = sorted(channel_scores.items(), key=lambda x: x[1], reverse=True)
        
        # Plot top N channels
        channels = [x[0] for x in sorted_channels[:n_top]]
        values = [x[1] for x in sorted_channels[:n_top]]
        
        # Create bar plot
        ax = axes[freq_idx]
        bars = ax.bar(range(len(channels)), values)
        
        # Customize plot
        ax.set_title(f'{freq_band}')
        ax.set_xticks(range(len(channels)))
        ax.set_xticklabels(channels, rotation=45, ha='right')
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}',
                   ha='center', va='bottom', rotation=90)
    
    # Add common ylabel
    fig.text(0.04, 0.5, 'Mean Absolute Br Score', va='center', rotation='vertical')
    
    plt.tight_layout()
    plt.show()
    
    # Print full ranking
    print("\nFull channel ranking by frequency band:")
    for freq_band in freq_bands:
        channel_scores = {}
        for ch_name in ch_names:
            if ch_name not in ignore_channels:
                abs_scores = []
                for mi in range(100, 501, 100):
                    val = scores[mi][freq_band][ch_name][layer]['Br']
                    if isinstance(val, (int, float)):
                        abs_scores.append(abs(val))
                if abs_scores:
                    channel_scores[ch_name] = np.mean(abs_scores)
        
        sorted_channels = sorted(channel_scores.items(), key=lambda x: x[1], reverse=True)
        print(f"\n{freq_band}:")
        for ch, score in sorted_channels:
            print(f"{ch}: {score:.3f}")

In [ ]:
# nest is topomap plot that shows Br scores. function takes layer as input, makes separate topomap for each frequency band and model index
# also possible to make 

In [ ]:
def plot_channel_importance(scores, layer, n_top=10, kind="Br"):
    # Create subplots for each frequency band
    fig, axes = plt.subplots(1, len(freq_bands), figsize=(20, 5))
    fig.suptitle(f'Channel importance based on absolute Br scores for layer: {layer}', fontsize=16)
    
    # Process each frequency band
    for freq_idx, freq_band in enumerate(freq_bands):
        # Aggregate absolute Br scores for each channel
        channel_scores = {}
        for ch_name in ch_names:
            if ch_name not in ignore_channels:
                abs_scores = []
                for mi in range(100, 501, 100):
                    val = scores[mi][freq_band][ch_name][layer][kind]
                    if isinstance(val, (int, float)):
                        abs_scores.append(abs(val))
                if abs_scores:
                    channel_scores[ch_name] = np.mean(abs_scores)
        
        # Sort channels by their scores
        sorted_channels = sorted(channel_scores.items(), key=lambda x: x[1], reverse=True)
        
        # Plot top N channels
        channels = [x[0] for x in sorted_channels[:n_top]]
        values = [x[1] for x in sorted_channels[:n_top]]
        
        # Create bar plot
        ax = axes[freq_idx]
        bars = ax.bar(range(len(channels)), values)
        
        # Customize plot
        ax.set_title(f'{freq_band}')
        ax.set_xticks(range(len(channels)))
        ax.set_xticklabels(channels, rotation=45, ha='right')
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}',
                   ha='center', va='bottom', rotation=90)
    
    # Add common ylabel
    fig.text(0.04, 0.5, 'Mean Absolute Br Score', va='center', rotation='vertical')
    
    plt.tight_layout()
    plt.show()
    
    # Print full ranking
    print("\nFull channel ranking by frequency band:")
    for freq_band in freq_bands:
        channel_scores = {}
        for ch_name in ch_names:
            if ch_name not in ignore_channels:
                abs_scores = []
                for mi in range(100, 501, 100):
                    val = scores[mi][freq_band][ch_name][layer][kind]
                    if isinstance(val, (int, float)):
                        abs_scores.append(abs(val))
                if abs_scores:
                    channel_scores[ch_name] = np.mean(abs_scores)
        
        sorted_channels = sorted(channel_scores.items(), key=lambda x: x[1], reverse=True)
        print(f"\n{freq_band}:")
        for ch, score in sorted_channels:
            print(f"{ch}: {score:.3f}")

In [ ]:
Br_scores

In [ ]:
for layer in layers:
    plot_channel_importance(Br_scores, layer)

In [ ]:
#for layer in layers:
#    plot_channel_importance(R2, layer, kind="mean_score")

In [ ]:
# next is topomap plot that shows Br scores. function takes layer as input, makes separate topomap for each frequency band and model index
# also possible to make 

In [ ]:
def load_info_subject(subject_index=2):

    file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
    epochs = mne.read_epochs(file_path)
    info_subj = epochs.info
      
    return info_subj

In [ ]:
info = load_info_subject(subject_index=2)

In [ ]:
def plot_layer_topomaps(scores, layer, info,vmin=None, vmax=None, use_abs=True):
    # Get model indices
    model_indices = list(range(100, 501, 100))
    
    # Create montage for 10-20 system

    
    # Create a figure
    fig, axes = plt.subplots(len(model_indices), len(freq_bands), 
                            figsize=(3*len(freq_bands), 3*len(model_indices)))
    fig.suptitle(f'Topographic distribution of Br scores for layer: {layer}', fontsize=16)
    
    # Create info object for topoplot

    
    # If vmin/vmax not provided, compute from all data
    if vmin is None or vmax is None:
        all_values = []
        for mi in model_indices:
            for freq_band in freq_bands:
                for ch_name in ch_names:
                   if ch_name not in ignore_channels:
                        val = scores[mi][freq_band][ch_name][layer]['Br']
                        if isinstance(val, (int, float)):
                            all_values.append(abs(val))

        if all_values:  # Only compute if all_values is not empty
            vmax = np.nanmax(all_values) if vmax is None else vmax
            if use_abs:
                vmin = 0
            else:
                vmin = -vmax
    # Create topoplot for each combination
    for i, mi in enumerate(model_indices):
        for j, freq_band in enumerate(freq_bands):
            ax = axes[i, j]
            
            # Collect values for all channels
            data = np.zeros(len(ch_names))
            for k, ch_name in enumerate(ch_names):
                if ch_name not in ignore_channels:
                    if use_abs:
                        val = np.abs(scores[mi][freq_band][ch_name][layer]['Br'])
                    else:

                        val = scores[mi][freq_band][ch_name][layer]['Br']
                    if isinstance(val, (int, float)):
                        data[k] = val
                    else:
                        data[k] = 0
                else:
                    data[k] = 0
            
            # Create topomap
            mne.viz.plot_topomap(data, info, show=False, names=ch_names, 
                           axes=ax, image_interp="nearest", vlim=(vmin,vmax))
            
            # Add titles
            if i == 0:
                ax.set_title(freq_band)
            if j == 0:
                ax.set_ylabel(f'MI={mi}')
    
    # Add colorbar
    cax = fig.add_axes([1, 0.1, 0.02, 0.8])
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(norm=norm, cmap='RdBu_r')
    fig.colorbar(sm, cax=cax, label='Br Score')
    
    plt.tight_layout()
    plt.show()

In [ ]:
layers

In [ ]:
for layer in layers:
    plot_layer_topomaps(Br_scores, layer, info)

In [ ]:
for layer in layers:
    plot_layer_topomaps(Br_scores, layer, info, use_abs=False)

In [ ]:
def get_channel_br_scores(scores, layer=None):
    """
    Get absolute Br scores for each channel organized by model index and frequency band.
    If layer is None, average scores across all layers.
    
    Args:
        scores: Dictionary containing Br scores
        layer: Model layer to analyze (default: None, which means average across all layers)
    
    Returns:
        Dictionary with structure: {model_index: {freq_band: {channel: abs_br_score}}}
    """
    # Initialize result dictionary
    result = {}
    
    # Process each model index
    for mi in range(100, 501, 100):
        result[mi] = {}
        
        # Process each frequency band
        for freq_band in freq_bands:
            result[mi][freq_band] = {}
            
            # Process each channel
            for ch_name in ch_names:
                if ch_name not in ignore_channels:
                    if layer is None:
                        # Average across all layers
                        scores_list = []
                        for l in layers:
                            val = scores[mi][freq_band][ch_name][l]['Br']
                            if isinstance(val, (int, float)):
                                scores_list.append(abs(val))
                        
                        if scores_list:  # Only compute mean if we have valid scores
                            result[mi][freq_band][ch_name] = np.mean(scores_list)
                        else:
                            result[mi][freq_band][ch_name] = 0.0
                    else:
                        # Use specified layer
                        val = scores[mi][freq_band][ch_name][layer]['Br']
                        if isinstance(val, (int, float)):
                            result[mi][freq_band][ch_name] = abs(val)
                        else:
                            result[mi][freq_band][ch_name] = 0.0
                        
    return result

output of trunket spatial filter looks extremely similar to channel ranking produced by explanaition function

In [ ]:
median_diff = np.load("/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/RCAV_parallel_perturbation_new/median_diff_per_channel.npy", allow_pickle=True).item()
mean_diff = np.load("/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/RCAV_parallel_perturbation_new/mean_diff_per_channel.npy", allow_pickle=True).item()

In [ ]:
result_Br = get_channel_br_scores(Br_scores, layer='trunk_net.spatial_filter')

In [ ]:
result_Br[300]["gamma"]

In [ ]:
result_Br.keys()

In [ ]:
def compute_rank_correlations(result, median_diff, amp_factor=1.5):
    """
    Compute rank correlations between Br scores and median differences for each model index and frequency band.
    
    Args:
        result: Dictionary containing Br scores
        median_diff: Dictionary containing median differences
    
    Returns:
        Dictionary containing correlations for each model index and frequency band
    """
    # Import scipy if not already imported
    import scipy.stats
    
    # Get model indices
    model_indices = list(range(100, 501, 100))
    
    # Initialize results dictionary
    correlations = {mi: {freq_band: None for freq_band in freq_bands} for mi in model_indices}
    
    # Process each model index
    for mi in model_indices:
        print(f"\nRank correlations for model index {mi-100}:")
        
        # Process each frequency band
        for freq_band in freq_bands:
            # Collect paired values
            paired_values = []
            for ch_name in ch_names:
                if ch_name not in ignore_channels:
                    br_score = result[mi][freq_band][ch_name]
                    med_diff = median_diff[mi][freq_band][amp_factor][ch_name]
                
                    # Only include if both values are valid numbers
                    if isinstance(br_score, (int, float)) and isinstance(med_diff, (int, float)):
                        paired_values.append((abs(br_score), abs(med_diff)))
            
            # Compute correlation if we have enough pairs
            if len(paired_values) > 1:
                br_scores, differences = zip(*paired_values)
                correlation,pval = scipy.stats.spearmanr(br_scores, differences)
                correlations[mi][freq_band] = correlation
                print(f"{freq_band}: {correlation:.3f}, p-value: {pval:.3e}")
            else:
                correlations[mi][freq_band] = float('nan')
                print(f"{freq_band}: N/A")
    
    return correlations

In [ ]:
def compute_channel_intersection(result, median_diff, k=10, amp_factor=1.5):
    """
    Compute intersection of top k channels between Br scores and median differences.
    
    Args:
        result: Dictionary containing Br scores
        median_diff: Dictionary containing median differences
        k: Number of top channels to consider (default: 10)
        amp_factor: Amplitude factor for median differences (default: 1.5)
    
    Returns:
        Dictionary containing intersection sizes for each model index and frequency band
    """
    # Get model indices
    model_indices = list(range(100, 501, 100))
    
    # Initialize results dictionary
    intersections = {mi: {freq_band: 0 for freq_band in freq_bands} for mi in model_indices}
    
    # Process each model index
    for mi in model_indices:
        print(f"\nIntersection sizes for model index {mi}:")
        
        # Process each frequency band
        for freq_band in freq_bands:
            # Get channel scores from result_Br
            br_scores = {}
            for ch_name in ch_names:
                if ch_name not in ignore_channels:
                    val = result[mi][freq_band][ch_name]  # Direct access to the score
                    if isinstance(val, (int, float)):
                        br_scores[ch_name] = abs(val)
            
            # Get channel scores for median differences
            med_scores = {}
            for ch_name in ch_names:
                if ch_name not in ignore_channels:
                    val = median_diff[mi][freq_band][amp_factor][ch_name]
                    if isinstance(val, (int, float)):
                        med_scores[ch_name] = abs(val)
            
            # Get top k channels for each metric
            top_br = set([x[0] for x in sorted(br_scores.items(), 
                                             key=lambda x: x[1], 
                                             reverse=True)[:k]])
            top_med = set([x[0] for x in sorted(med_scores.items(), 
                                              key=lambda x: x[1], 
                                              reverse=True)[:k]])
            
            # Compute intersection size
            intersection_size = len(top_br.intersection(top_med))
            intersections[mi][freq_band] = intersection_size
            
            print(f"{freq_band}: {intersection_size}/{k}")
            #print(f"Br top {k}: {', '.join(sorted(top_br))}")
            #print(f"Med top {k}: {', '.join(sorted(top_med))}")
            print(f"Intersection: {', '.join(sorted(top_br.intersection(top_med)))}")
    
    return intersections

In [ ]:
compute_rank_correlations(result_Br, median_diff)

In [ ]:
all_rank_correlations = {}
for layer in layers:
    print(layer)
    result_Br = get_channel_br_scores(Br_scores, layer=layer)
    all_rank_correlations[layer] = compute_rank_correlations(result_Br, median_diff)
    

In [ ]:
def get_top_channels(result_Br, n_top=5):
    # Get model indices
    model_indices = list(range(100, 501, 100))
    
    # Process each model index and frequency band
    for mi in model_indices:
        print(f"\nModel Index {mi}:")
        for freq_band in freq_bands:
            # Get channel scores for this model index and frequency band
            channel_scores = {}
            for ch_name in ch_names:
                if ch_name not in ignore_channels:
                    val = result_Br[mi][freq_band][ch_name]
                    if isinstance(val, (int, float)):
                        channel_scores[ch_name] = abs(val)
            
            # Sort channels by absolute score and get top n
            top_channels = sorted(channel_scores.items(), key=lambda x: x[1], reverse=True)[:n_top]
            
            # Print results
            print(f"\n{freq_band}:")
            for ch, score in top_channels:
                print(f"  {ch}: {score:.3f}")

# Call the function
get_top_channels(result_Br)

In [ ]:
def plot_rank_correlations():
    # Create figure with subplots - one row per frequency band
    fig, axes = plt.subplots(len(freq_bands), 1, figsize=(16, 3*len(freq_bands)), sharex=True, sharey=True)
    fig.suptitle('Rank Correlations between Br Scores and Channel importances', fontsize=24)
    
    # Colors for different model indices
    model_indices = list(range(100, 401, 100))
    colors = plt.cm.viridis(np.linspace(0, 1, len(model_indices)))
    
    # Plot for each frequency band
    for freq_idx, freq_band in enumerate(freq_bands):
        ax = axes[freq_idx]
        
        # Plot lines for each model index
        for mi_idx, mi in enumerate(model_indices):
            correlations = []
            for layer in layers:
                result_Br = get_channel_br_scores(Br_scores, layer=layer)
                # Get correlation for this layer
                paired_values = []
                for ch_name in ch_names:
                    if ch_name not in ignore_channels:
                        br_score = result_Br[mi][freq_band][ch_name]
                        med_diff = median_diff[mi][freq_band][1.5][ch_name]
                        if isinstance(br_score, (int, float)) and isinstance(med_diff, (int, float)):
                            paired_values.append((abs(br_score), abs(med_diff)))
                
                if len(paired_values) > 1:
                    br_scores, differences = zip(*paired_values)
                    correlation, _ = scipy.stats.spearmanr(br_scores, differences)
                    correlations.append(correlation)
                else:
                    correlations.append(np.nan)
            
            # Plot line
            ax.plot(range(len(layers)), correlations, 'o-', 
                   label=f'Time index={mi-100}', color=colors[mi_idx], alpha=0.8,
                   linewidth=2, markersize=8)
        
        # Add horizontal line at y=0
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        
        # Customize subplot
        ax.set_title(f'Frequency Band: {freq_band}', fontsize=18)
        ax.grid(True, alpha=0.3)
        ax.set_xticks(range(len(layers)))
        ax.set_xticklabels(layer_labels, rotation=30, ha='right', fontsize=16)
        ax.set_ylabel('Rank Correlation', fontsize=16)
        ax.set_ylim(-1, 1)  # correlation range is [-1, 1]
        ax.tick_params(axis='y', labelsize=15)
        
        # Add legend only for the first subplot
        if freq_idx == 0:
            ax.legend(bbox_to_anchor=(0.02, 1), loc='upper left', fontsize=15)
    
    plt.tight_layout()
    return fig

In [ ]:
fig = plot_rank_correlations()

In [ ]:
fig.savefig(f"rank_correlations.png", dpi=300, bbox_inches='tight')